# Understanding Model Structure from config.json

> In the previous sections, we have been building from scratch: writing Tokenizers, constructing Embeddings, stacking Transformer Blocks -- all structural parameters hardcoded in Python code, so changing a hidden dimension requires rewriting the code. But real-world large models are not organized and distributed this way.

> This section opens the SmolLM2-135M repository to see what files a modern LLM actually consists of, what each file controls, and how to use these files to load and run a model. We switch from "writing code to define a model" to "reading configs to describe a model."

A typical HuggingFace model repository (such as [SmolLM2-135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M/tree/main)) contains weight files (.safetensors) that make up only half of it. The other half consists of several JSON configuration files, each responsible for its own domain:

- config.json controls what the model looks like -- how many layers, how wide, how many heads
- tokenizer_config.json controls how text becomes numbers -- whether to add BOS/EOS, maximum tokenization length
- tokenizer.json stores the BPE vocabulary and merge rules, serving as the "data file" for tokenizer_config.json
- generation_config.json controls how the model outputs -- temperature, top_p, top_k

Together, these files form a complete model specification.

Every parameter in config.json corresponds to a concrete PyTorch module.

## 1. Repository File Map

A HuggingFace model repository typically contains these files. config.json is required; the rest depend on the model type and configuration approach.

In [ ]:
files = [
    ("config.json",          "Required", "Model structure: layers, dimensions, heads, activation, etc."),
    ("tokenizer_config.json", "Standard", "Tokenizer behavior: special tokens, max length, truncation/padding"),
    ("generation_config.json","Standard", "Generation strategy: temperature, top_p, top_k, repetition_penalty"),
    ("tokenizer.json",        "Standard", "Tokenizer model file (BPE vocab + merge rules), usually a few MB"),
    ("special_tokens_map.json","Optional", "Name-to-ID mapping for special tokens"),
    ("vocab.json",            "Partial", "BPE vocabulary (e.g., GPT-2), token string to ID"),
    ("merges.txt",            "Partial", "BPE merge rules (e.g., GPT-2), ordered by priority"),
]

print(f"{'Filename':<28} {'Status':<10} {'Purpose'}")
print("-" * 80)
for name, required, purpose in files:
    print(f"{name:<28} {required:<10} {purpose}")

## 2. config.json -- Turning Structure Parameters into PyTorch Modules

Below is the real config.json of SmolLM2-135M. In each subsection that follows, instead of just printing the meaning of these values, we **build the corresponding PyTorch modules using them**, verify shapes, and count parameters.

In [ ]:
config = {
    "architectures": ["LlamaForCausalLM"],
    "hidden_size": 576,
    "intermediate_size": 1536,
    "num_attention_heads": 9,
    "num_key_value_heads": 3,
    "num_hidden_layers": 30,
    "vocab_size": 49152,
    "max_position_embeddings": 8192,
    "hidden_act": "silu",
    "rms_norm_eps": 1e-05,
    "rope_theta": 100000,
    "tie_word_embeddings": True,
    "attention_bias": False,
}

V, D, L, H, KV, FF = (config[k] for k in (
    "vocab_size", "hidden_size", "num_hidden_layers",
    "num_attention_heads", "num_key_value_heads", "intermediate_size"))
head_dim = D // H

### 2.1 TransformerBlock -- Assembling a Complete Block from config

The parameters in config.json ultimately go into a TransformerBlock: Attention (GQA) + FFN (SwiGLU) + RMSNorm.
Below we use PyTorch's built-in modules (`nn.Linear`, `nn.RMSNorm`) directly, wire them together with the config values, and print the structure.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    """TransformerBlock: RMSNorm -> GQA Attention -> RMSNorm -> SwiGLU FFN"""
    def __init__(self, config):
        super().__init__()
        d = config['hidden_size']
        ff = config['intermediate_size']
        h = config['num_attention_heads']
        kv = config['num_key_value_heads']
        hd = d // h
        bias = config['attention_bias']

        self.attn_norm = nn.RMSNorm(d, eps=config['rms_norm_eps'])
        self.ffn_norm  = nn.RMSNorm(d, eps=config['rms_norm_eps'])
        # Attention: four projections
        self.q_proj = nn.Linear(d, h * hd, bias=bias)
        self.k_proj = nn.Linear(d, kv * hd, bias=bias)
        self.v_proj = nn.Linear(d, kv * hd, bias=bias)
        self.o_proj = nn.Linear(h * hd, d, bias=bias)
        # FFN: three projections
        self.gate = nn.Linear(d, ff, bias=False)
        self.up   = nn.Linear(d, ff, bias=False)
        self.down = nn.Linear(ff, d, bias=False)

        self.n_heads = h
        self.n_kv_heads = kv
        self.head_dim = hd

    def forward(self, x):
        # Attention (simplified, no causal mask or RoPE)
        residual = x
        x = self.attn_norm(x)
        B, T, D = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        # GQA broadcast
        k = k.repeat_interleave(self.n_heads // self.n_kv_heads, dim=1)
        v = v.repeat_interleave(self.n_heads // self.n_kv_heads, dim=1)
        scale = self.head_dim ** -0.5
        attn = (q @ k.transpose(-2, -1)) * scale
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, D)
        x = residual + self.o_proj(out)
        # FFN
        residual = x
        x = self.ffn_norm(x)
        x = residual + self.down(F.silu(self.gate(x)) * self.up(x))
        return x

# Build one Block using SmolLM2's config
block = TransformerBlock(config)
print("=== Complete structure of a single TransformerBlock ===")
print(block)
block_params = sum(p.numel() for p in block.parameters())
print(f"\nParameters in this Block: {block_params:,}  ({block_params/1e6:.2f}M)")

### 2.2 Attention -- GQA Makes K/V Projections Smaller Than Q

num_attention_heads=9, num_key_value_heads=3, attention_bias=false.
The Q projection is [576, 9x64], while K/V projections are [576, 3x64] -- K and V have only 1/3 the parameters of Q.
Let's build the four projection matrices and verify by inspecting their shapes directly.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

W_q = nn.Linear(D, H * head_dim, bias=False)
W_k = nn.Linear(D, KV * head_dim, bias=False)
W_v = nn.Linear(D, KV * head_dim, bias=False)
W_o = nn.Linear(H * head_dim, D, bias=False)

x = torch.randn(2, 16, D)  # Simulated hidden states
q = W_q(x).view(2, 16, H, head_dim).transpose(1, 2)   # [2, 9, 16, 64]
k = W_k(x).view(2, 16, KV, head_dim).transpose(1, 2)  # [2, 3, 16, 64]
v = W_v(x).view(2, 16, KV, head_dim).transpose(1, 2)  # [2, 3, 16, 64]

print(f"Input: [2, 16, {D}]")
print(f"Q projection: {list(W_q.weight.shape)} -> Q shape: {list(q.shape)}")
print(f"K projection: {list(W_k.weight.shape)} -> K shape: {list(k.shape)}")
print(f"V projection: {list(W_v.weight.shape)} -> V shape: {list(v.shape)}")
print(f"O projection: {list(W_o.weight.shape)}")
print()
q_p = sum(p.numel() for p in W_q.parameters())
k_p = sum(p.numel() for p in W_k.parameters())
print(f"Q params: {q_p:,}   K params: {k_p:,}   K/Q = {k_p/q_p:.2f}")
print(f"If this were MHA (Q=K=V=9): K params would also be {q_p:,}")
print(f"GQA saves {(q_p - k_p) * 30:,.0f} K+V params across 30 layers")

The core of GQA is "broadcasting" KV heads to Q heads during the Attention computation. The following small example simulates this process:

In [ ]:
groups = H // KV  # 3 Q heads per group
k_repeated = k.repeat_interleave(groups, dim=1)  # [2, 3, 16, 64] -> [2, 9, 16, 64]
v_repeated = v.repeat_interleave(groups, dim=1)

# Now Q and K have the same number of heads, so we can compute Attention normally
scale = head_dim ** -0.5
attn_weights = (q @ k_repeated.transpose(-2, -1)) * scale  # [2, 9, 16, 16]

print(f"K original shape: {list(k.shape)}  -> repeat_interleave({groups}) -> {list(k_repeated.shape)}")
print(f"V original shape: {list(v.shape)}  -> repeat_interleave({groups}) -> {list(v_repeated.shape)}")
print(f"QK^T result:      {list(attn_weights.shape)}")
print(f"\nGroup mapping:")
for kv_idx in range(KV):
    q_idx = list(range(kv_idx * groups, (kv_idx + 1) * groups))
    print(f"  KV[{kv_idx}] -> Q{q_idx}")

### 2.3 FFN -- SwiGLU's Three-Weight Structure

Llama's FFN differs from the simplest two-layer FFN in Section 05: it has three weight matrices (gate, up, down),
and uses SiLU as the activation. intermediate_size=1536 is the result of 576 x 8/3.
Let's build it, run a forward pass, and verify the shape transformations.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

import torch

class LlamaFFN(nn.Module):
    """Llama-style SwiGLU FFN: gate for gating, up for projection, down to project back"""
    def __init__(self, dim, intermediate_dim):
        super().__init__()
        self.gate = nn.Linear(dim, intermediate_dim, bias=False)
        self.up   = nn.Linear(dim, intermediate_dim, bias=False)
        self.down = nn.Linear(intermediate_dim, dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

ffn = LlamaFFN(D, FF)
x = torch.randn(2, 16, D)
out = ffn(x)

gate_p = sum(p.numel() for p in ffn.gate.parameters())
up_p   = sum(p.numel() for p in ffn.up.parameters())
down_p = sum(p.numel() for p in ffn.down.parameters())

print(f"LlamaFFN structure:")
print(f"  gate: {list(ffn.gate.weight.shape)}  ({gate_p:,} params)")
print(f"  up:   {list(ffn.up.weight.shape)}  ({up_p:,} params)")
print(f"  down: {list(ffn.down.weight.shape)}  ({down_p:,} params)")
print(f"  Total: {gate_p + up_p + down_p:,} params")
print(f"\nForward: [2, 16, {D}] -> gate/up -> [2, 16, {FF}] -> SiLU * up -> down -> [2, 16, {D}]")
print(f"Input shape: {list(x.shape)}  Output shape: {list(out.shape)}")
print(f"Comparison with Section 05: two-weight FFN ({D}->{D*4}->{D}), here three-weight ({D}->{FF}->{D}), with an added gate")

### 2.4 RMSNorm -- Scaling Without Shifting

Section 05 used LayerNorm (shift + scale). Modern LLMs almost universally use RMSNorm -- it only scales,
skipping the shift step. The config specifies rms_norm_eps=1e-05. PyTorch has included nn.RMSNorm since version 1.13.
Let's print its structure and compare it with nn.LayerNorm.

In [ ]:
# Using PyTorch's built-in RMSNorm and LayerNorm directly
import torch
import torch.nn as nn

rn = nn.RMSNorm(D, eps=config['rms_norm_eps'])
ln = nn.LayerNorm(D, eps=config['rms_norm_eps'])

print("=== nn.RMSNorm structure ===")
print(rn)
print("\n=== nn.LayerNorm structure ===")
print(ln)

# Same random input
x = torch.randn(4, 8, D)
with torch.no_grad():
    ln_out = ln(x)
    rn_out = rn(x)

print(f"\nLayerNorm:  weight + bias = {sum(p.numel() for p in ln.parameters())} params")
print(f"RMSNorm:   weight only    = {sum(p.numel() for p in rn.parameters())} params")
print(f"\nBefore normalization:  mean={x.mean():.3f}, std={x.std():.3f}")
print(f"LayerNorm: mean={ln_out.mean():.6f}, std={ln_out.std():.3f}  <- mean is 0")
print(f"RMSNorm:   mean={rn_out.mean():.4f}, std={rn_out.std():.3f}  <- mean is NOT 0")
print(f"\nSaves {D} bias params per layer, {D * 30:,} total across 30 layers")

### 2.5 Counting Total Parameters

Now we have the dimensions of all components. Let's verify the 135M figure in two steps:
First, compute item by item using formulas (theory), then build the complete model and use `sum(p.numel())` (actual), and see if they match.

In [ ]:
# ========== Theoretical calculation: item by item using formulas ==========
# Per-layer Attention: Q, K, V, O four projections
import torch.nn as nn
import torch.nn.functional as F

q_p  = D * H * head_dim       # Q: [D, H*head_dim]
k_p  = D * KV * head_dim      # K: [D, KV*head_dim]
v_p  = D * KV * head_dim      # V: [D, KV*head_dim]
o_p  = H * head_dim * D       # O: [H*head_dim, D]
attn_p = q_p + k_p + v_p + o_p

# Per-layer FFN: gate, up, down three projections
ffn_p = 3 * D * FF

# Per-layer Norm: 2 RMSNorm, each with only D weights
norm_p = 2 * D

per_layer = attn_p + ffn_p + norm_p
total_theory = V * D + L * per_layer + D  # Embedding + 30 layers + final Norm

print("========== Theoretical Calculation (formulas) ==========")
print(f"Per-layer Attention (Q+K+V+O): {attn_p:>10,}")
print(f"Per-layer FFN (gate+up+down):   {ffn_p:>10,}")
print(f"Per-layer RMSNorm x 2:          {norm_p:>10,}")
print(f"Per-layer subtotal:             {per_layer:>10,} ~ {per_layer/1e6:.2f}M")
print(f"\nEmbedding ({V}x{D}):       {V*D:>10,} ~ {V*D/1e6:.1f}M")
print(f"{L} layers of Blocks:        {L*per_layer:>10,} ~ {L*per_layer/1e6:.1f}M")
print(f"Final RMSNorm:                  {D:>10,}")
print(f"Theoretical total:              {total_theory:>10,} ~ {total_theory/1e6:.1f}M")

# ========== Actual calculation: build the full model, use sum(p.numel()) ==========
class SmolLM2ConfigModel(nn.Module):
    """Assemble SmolLM2 from config: Embedding -> 30x Block -> RMSNorm -> output projection"""
    def __init__(self, config):
        super().__init__()
        d, V = config['hidden_size'], config['vocab_size']
        self.embed = nn.Embedding(V, d)
        self.blocks = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config['num_hidden_layers'])])
        self.final_norm = nn.RMSNorm(d, eps=config['rms_norm_eps'])

    def forward(self, x):
        x = self.embed(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.final_norm(x)
        # tie_word_embeddings=True: output projection reuses embed.weight, no separate lm_head
        return F.linear(x, self.embed.weight)

model = SmolLM2ConfigModel(config)
total_real = sum(p.numel() for p in model.parameters())

print("\n========== Actual Calculation (sum(p.numel())) ==========")
emb_real = sum(p.numel() for p in model.embed.parameters())
blocks_real = sum(p.numel() for p in model.blocks.parameters())
norm_real = sum(p.numel() for p in model.final_norm.parameters())
print(f"Embedding:     {emb_real:>10,}  ({V}x{D})")
print(f"30x Block:     {blocks_real:>10,}  ({L}x{per_layer:,})")
print(f"final_norm:    {norm_real:>10,}  ({D})")
print(f"{'─'*45}")
print(f"Actual total:  {total_real:>10,} ~ {total_real/1e6:.1f}M")

print(f"\n========== Comparison ==========")
print(f"Theoretical: {total_theory:,}")
print(f"Actual:      {total_real:,}")
print(f"Match:       {total_theory == total_real}")
if total_theory == total_real:
    print("The theoretical formulas and PyTorch's actual parameters match perfectly")
else:
    print(f"Difference of {abs(total_theory - total_real):,}, please check")

Where the parameters are spent:

In [ ]:
import matplotlib.pyplot as plt

emb_s = V * D
attn_s = attn_p * L
ffn_s = ffn_p * L
norm_s = norm_p * L

labels = [f'Embedding\n{emb_s/1e6:.1f}M', f'Attention x{L}\n{attn_s/1e6:.1f}M',
          f'FFN x{L}\n{ffn_s/1e6:.1f}M', f'RMSNorm x{L}\n{norm_s/1e6:.2f}M']
sizes = [emb_s, attn_s, ffn_s, norm_s]

plt.figure(figsize=(6, 5))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90,
        colors=['#5DADE2','#F5B041','#E74C3C','#58D68D'])
plt.title(f'SmolLM2-135M Parameter Distribution')
plt.tight_layout()
plt.show()

## 3. tokenizer_config.json -- Controlling How Text Gets "Translated" into Numbers

config.json only governs the model's internals. Before text enters the model and after it leaves, tokenizer_config.json and tokenizer.json take over.
tokenizer_config.json defines behavioral parameters such as "what are the special tokens", "maximum tokenization length", and "whether to add BOS/EOS".

Below we use SmolLM2's tokenizer config (simplified version) to demonstrate what these parameters actually affect.

In [ ]:
import json

tokenizer_config = {
    "add_bos_token": True,
    "add_eos_token": True,
    "bos_token": "<|im_start|>",
    "eos_token": "<|im_end|>",
    "pad_token": "<|im_end|>",
    "model_max_length": 8192,
    "truncation_side": "right",
    "padding_side": "right",
}

print("tokenizer_config.json (simplified):")
print(json.dumps(tokenizer_config, indent=2, ensure_ascii=False))

The parameters in tokenizer_config directly affect the result of encoding.
Let's use a small vocabulary to simulate the encoding process and show how BOS/EOS insertion, padding, and truncation work:

In [ ]:
# Simulate a small vocabulary (SmolLM2 has 49152 tokens, we use 10 for illustration)
mini_vocab = {"<|im_start|>": 0, "<|im_end|>": 1, "I": 2, "love": 3, "ML": 4, "<unk>": 5, ".": 6}

def simulate_encode(text, cfg):
    """Simulate tokenizer encode behavior: split tokens + add special tokens"""
    # Simplified tokenization: one token per word (real tokenizer uses BPE, but the behavioral pattern is the same)
    tokens = text.split()
    ids = [mini_vocab.get(t, -1) for t in tokens]
    if cfg.get("add_bos_token"):
        ids = [mini_vocab[cfg["bos_token"]]] + ids
    if cfg.get("add_eos_token"):
        ids = ids + [mini_vocab[cfg["eos_token"]]]
    return ids

# Encode results under different settings
text = "I love ML"

cfg_with_both = {"add_bos_token": True, "add_eos_token": True, "bos_token": "<|im_start|>", "eos_token": "<|im_end|>"}
cfg_without   = {"add_bos_token": False, "add_eos_token": False}
cfg_bos_only  = {"add_bos_token": True, "add_eos_token": False, "bos_token": "<|im_start|>", "eos_token": "<|im_end|>"}

print(f"Original text: '{text}'\n")
print(f"add_bos=True, add_eos=True:   {simulate_encode(text, cfg_with_both)}")
print(f"add_bos=False, add_eos=False: {simulate_encode(text, cfg_without)}")
print(f"add_bos=True, add_eos=False:  {simulate_encode(text, cfg_bos_only)}")
print(f"\nKey observation: BOS/EOS insertion is controlled by tokenizer_config, not by the model itself.")

In [ ]:
# Padding demo: how sentences of different lengths get aligned
sentences = ["I love ML", "Hi", "deep learning is fun"]
encoded = [simulate_encode(s, cfg_with_both) for s in sentences]
max_len = max(len(e) for e in encoded)

print("Padding demo (pad_token_id = 1 = <|im_end|>):\n")
for i, (s, ids) in enumerate(zip(sentences, encoded)):
    pad_len = max_len - len(ids)
    padded = ids + [1] * pad_len  # Pad with pad_token_id
    print(f"  '{s}':  {ids} -> padded: {padded}")

print(f"\npadding_side = '{tokenizer_config['padding_side']}' -> padding on the right")
print(f"model_max_length = {tokenizer_config['model_max_length']} -> sequences exceeding this will be truncated")

## 4. generation_config.json -- Controlling How the Model "Speaks"

After the model produces logits for all tokens at each position, how is the next token selected?
That is what generation_config.json controls. The three parameters temperature, top_p, and top_k determine the sampling strategy.
Their individual effects can be understood through a concrete example.

In [ ]:
import json

generation_config = {
    "temperature": 0.6,
    "top_p": 0.9,
    "top_k": 50,
    "max_new_tokens": 256,
    "do_sample": True,
    "repetition_penalty": 1.1,
    "eos_token_id": 0,
    "pad_token_id": 0,
}

print("generation_config.json:")
print(json.dumps(generation_config, indent=2, ensure_ascii=False))

### 4.1 temperature -- Adjusting "Determinism"

temperature divides the logits before softmax.
When T < 1, the probability distribution becomes sharper (high-probability tokens become even more likely, the model is more deterministic).
When T > 1, the distribution becomes flatter (more tokens have a chance of being selected, output is more random).

Let's construct a small vocabulary with a set of logits and see how temperature changes the actual probability distribution.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import torch

tokens = ["the", "is", "at", "in", "on", "to", "and", "or", "but", "so"]
logits = torch.tensor([3.2, 2.8, 2.5, 2.0, 1.5, 1.2, 0.9, 0.6, 0.3, 0.1])

temperatures = [0.3, 0.6, 1.0, 2.0]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

for ax, T in zip(axes, temperatures):
    probs = torch.softmax(logits / T, dim=-1).numpy()
    ax.bar(range(len(tokens)), probs, color=['#E74C3C' if p == probs.max() else '#BDC3C7' for p in probs])
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"T={T}")
    ax.set_ylabel("Probability")
    # Annotate the top 2 probabilities
    top2 = np.argsort(probs)[-2:]
    ax.annotate(f"{probs[top2[0]]:.2f}", (top2[0], probs[top2[0]]), ha='center', va='bottom', fontsize=9)
    ax.annotate(f"{probs[top2[1]]:.2f}", (top2[1], probs[top2[1]]), ha='center', va='bottom', fontsize=9)

plt.suptitle("Effect of temperature on probability distribution", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Key observation: At T=0.3, the highest-probability token almost monopolizes; at T=2.0, probabilities become nearly uniform.")
print(f"  SmolLM2 default T={generation_config['temperature']}, which is conservative and suitable for factual tasks.")

### 4.2 top_p (nucleus sampling) -- Keep Only Tokens Whose Cumulative Probability Exceeds p

top_p does not simply take the top k tokens by probability. Instead, it accumulates probabilities from highest to lowest until the total reaches p.
The result is that the number of selected tokens is not fixed -- fewer when probability is concentrated at the top, more when the distribution is flat.

In [ ]:
import torch
import matplotlib.pyplot as plt

probs = torch.softmax(logits, dim=-1)
sorted_probs, sorted_indices = torch.sort(probs, descending=True)
cumsum = torch.cumsum(sorted_probs, dim=-1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left plot: cumulative probability
ax1.bar(range(len(tokens)), sorted_probs.numpy(), color='#3498DB', alpha=0.7)
ax1.plot(range(len(tokens)), cumsum.numpy(), 'o-', color='#E74C3C', linewidth=2, markersize=6)
ax1.axhline(y=0.9, color='gray', linestyle='--', alpha=0.7, label='top_p=0.9')
ax1.set_xticks(range(len(tokens)))
ax1.set_xticklabels([tokens[i] for i in sorted_indices])
ax1.set_ylabel("Probability / Cumulative Probability")
ax1.legend()

# Right plot: tokens kept after top_p=0.9
mask = cumsum <= 0.9
keep_n = mask.sum().item() + 1  # +1 because we need to include the first token that exceeds the threshold
keep_indices = sorted_indices[:keep_n]
keep_probs = probs[keep_indices]
filtered = torch.zeros_like(probs)
filtered[keep_indices] = keep_probs / keep_probs.sum()

ax2.bar(range(len(tokens)), filtered.numpy(),
        color=['#27AE60' if i in keep_indices else '#BDC3C7' for i in range(len(tokens))])
ax2.set_xticks(range(len(tokens)))
ax2.set_xticklabels(tokens)
ax2.set_ylabel("Renormalized Probability")
ax2.set_title(f"top_p=0.9: keeps {keep_n} tokens, discards the rest")

plt.tight_layout()
plt.show()

print(f"top_p=0.9: kept the top {keep_n} tokens ({[tokens[i] for i in sorted_indices[:keep_n]]})")
print(f"Discarded tokens: {[tokens[i] for i in sorted_indices[keep_n:]]}")

### 4.3 top_k -- The Simplest Truncation

Keep only the k tokens with the highest probability. SmolLM2's top_k=50
retains only 0.1% of the options from a vocabulary of 49152.
top_k is crude but effective -- it ensures no completely absurd token gets selected.

In [ ]:
import torch
import matplotlib.pyplot as plt

K_list = [3, 5, 10, len(tokens)]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

for ax, K in zip(axes, K_list):
    topk_indices = sorted_indices[:K]
    topk_probs = probs[topk_indices]
    topk_probs = topk_probs / topk_probs.sum()
    display = torch.zeros_like(probs)
    display[topk_indices] = topk_probs
    colors = ['#27AE60' if i in topk_indices else '#BDC3C7' for i in range(len(tokens))]
    ax.bar(range(len(tokens)), display.numpy(), color=colors)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens)
    ax.set_ylim(0, 1.05)
    ax.set_title(f"top_k={K}")

plt.suptitle("Effect of top_k on candidate token truncation", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"SmolLM2 vocabulary has {V:,} tokens, top_k={generation_config['top_k']} keeps only the top {generation_config['top_k']}")
print(f"Retention ratio: {generation_config['top_k'] / V * 100:.2f}%")

### 4.4 Combined Effect of temperature + top_p + top_k

During actual generation, all three parameters act simultaneously: first divide by temperature to change the distribution shape, then truncate to the top k via top_k, and finally filter the long tail with top_p.
Let's simulate different combinations and see which tokens ultimately "survive."

In [ ]:
import torch

def sample_filter(all_tokens, logits, T=1.0, top_k=None, top_p=None):
    """Apply temperature, top_k, top_p and return the list of kept tokens"""
    scaled = logits / T
    probs = torch.softmax(scaled, dim=-1)
    allowed = torch.ones_like(probs, dtype=torch.bool)

    # top_k cannot exceed vocabulary size
    k = min(top_k, len(all_tokens)) if top_k is not None else None
    if k is not None:
        topk_vals, _ = torch.topk(probs, k)
        threshold = topk_vals[-1]
        allowed = allowed & (probs >= threshold)

    if top_p is not None:
        sorted_p, sorted_i = torch.sort(probs, descending=True)
        cum = torch.cumsum(sorted_p, dim=-1)
        n_keep = min((cum <= top_p).sum().item() + 1, len(all_tokens))
        keep_i = sorted_i[:n_keep]
        mask = torch.zeros_like(probs, dtype=torch.bool)
        mask[keep_i] = True
        allowed = allowed & mask

    return [all_tokens[i] for i in range(len(all_tokens)) if allowed[i]]

# Three configurations (top_k adjusted to fit the demo vocabulary size)
configs = [
    ("Conservative", 0.3, 3, 0.85),
    ("Greedy (T->0)", 0.001, None, None),
    ("Creative", 1.2, None, 0.95),
]

logits_np = logits.clone()
print(f"{'Config':<20} {'Kept tokens'}")
print("-" * 50)
for name, T, tk, tp in configs:
    kept = sample_filter(tokens, logits_np, T=T, top_k=tk, top_p=tp)
    print(f"{name:<20} {kept}")

print(f"\nKey observation: With all three parameters combined, the final candidate tokens are their intersection.")
print(f"  Greedy mode: only the highest-probability token survives = deterministic generation")
print(f"  Creative mode: more candidates retained = more diverse output")

## 5. Connecting the Three Configs: The Complete Journey of a Request

Now let's connect config.json, tokenizer_config.json, and generation_config.json to see: at each stage along the complete path from text input to output, which config file is in charge.

In [ ]:
print("Input text: 'I love ML'")
print()
print("  +-------------------------------+")
print("  |  tokenizer_config.json        |")
print("  |  * add_bos_token = True       |  <- automatically adds <|im_start|>")
print("  |  * add_eos_token = True       |  <- automatically adds <|im_end|>")
print("  |  * model_max_length = 8192    |  <- truncate if exceeded")
print("  +---------------+---------------+")
print("                  | token IDs")
print("                  v")
print("  +-------------------------------+")
print("  |  config.json                  |")
print("  |  * hidden_size = 576          |  <- Embedding vector dimension")
print("  |  * num_hidden_layers = 30     |  <- passes through 30 Block layers")
print("  |  * num_attention_heads = 9    |  <- GQA attention")
print("  |  * intermediate_size = 1536   |  <- FFN intermediate dimension")
print("  |  * vocab_size = 49152         |  <- outputs 49152 logits")
print("  +---------------+---------------+")
print("                  | logits [1, 49152]")
print("                  v")
print("  +-------------------------------+")
print("  |  generation_config.json       |")
print("  |  * temperature = 0.6          |  <- adjusts sharpness")
print("  |  * top_k = 50                 |  <- keep only top 50")
print("  |  * top_p = 0.9                |  <- cumulative probability to 90%")
print("  |  * max_new_tokens = 256       |  <- generate at most 256 tokens")
print("  +---------------+---------------+")
print("                  | selected token ID")
print("                  v")
print("           Output: 'deep learning is fun'")
print()
print("The three configs have clear divisions of labor:")
print("tokenizer_config handles input, config handles computation, generation_config handles output.")
print("Changing any one of them changes the model's behavior -- this is why reading")
print("different models' configs helps you understand their design philosophy.")

## Summary

- [ ] A HuggingFace model repository has 3 core JSON configs: config.json, tokenizer_config.json, generation_config.json
- [ ] Every number in config.json corresponds to a concrete PyTorch module: Embedding, Q/K/V/O, FFN, RMSNorm
- [ ] GQA saves parameters and memory by reducing the number of KV heads: K projection parameters are only 1/3 of Q
- [ ] Llama's FFN has an additional gate matrix compared to the teaching version (three weights vs two weights), using SiLU for gating
- [ ] RMSNorm eliminates LayerNorm's bias and shift step, saving D parameters per layer
- [ ] tokenizer_config.json controls BOS/EOS insertion, padding direction, and maximum length
- [ ] The three core parameters in generation_config.json (temperature, top_p, top_k) jointly determine sampling behavior
- [ ] Changing any one of these config files changes the model's behavior

Once you learn to read these config files, you have a complete specification for any model. Open any new model on HuggingFace: first check config.json for structural dimensions, then tokenizer_config.json for special tokens, and finally generation_config.json for sampling strategy -- after reading all three, the model's design philosophy becomes clear.

## Exercises

> You can ask AI for help with reasoning, but it is not recommended to have AI "complete the exercise for you" directly.

**Exercise 1: Compare generation_config of two models**

Find two models with different purposes on HuggingFace (e.g., a general-purpose chat model and a code generation model), compare their generation_config.json, identify three key differences, and explain why.

Hint: Code generation typically has a lower temperature and higher repetition_penalty.

In [ ]:
# Exercise 1: Compare generation_config of two models
# Example: Open these two URLs in a browser
#   https://huggingface.co/HuggingFaceTB/SmolLM2-135M/blob/main/generation_config.json
#   https://huggingface.co/Qwen/Qwen2.5-0.5B/blob/main/generation_config.json

# TODO: Write out three differences and your explanations
diff_1 = """Difference 1 and your explanation"""
diff_2 = """Difference 2 and your explanation"""
diff_3 = """Difference 3 and your explanation"""

print("Complete Exercise 1: Replace the placeholders above, then uncomment the assert below to verify")
# assert all(not d.startswith('"""') for d in [diff_1, diff_2, diff_3]), "Please replace the placeholders first"
# print("Exercise 1 passed")

**Exercise 2: Build your own config-to-module mapping**

Modify the config parameters below, then run the cell to see how the parameter count and shapes change.
For example, change hidden_size from 576 to 768 and see what the total parameter count becomes.

Hint: Pay attention to which components' parameter counts are affected when hidden_size changes.

In [ ]:
# Exercise 2: Modify config and observe parameter count changes
my_config = {
    "vocab_size": 49152,
    "hidden_size": 576,        # TODO: Try changing to 768
    "num_hidden_layers": 30,   # TODO: Try changing to 12
    "num_attention_heads": 9,
    "num_key_value_heads": 3,
    "intermediate_size": 1536, # TODO: Try changing to 2048
}

V2, D2, L2, H2, KV2, FF2 = (my_config[k] for k in (
    "vocab_size", "hidden_size", "num_hidden_layers",
    "num_attention_heads", "num_key_value_heads", "intermediate_size"))
hd2 = D2 // H2

emb_p = V2 * D2
attn_p = D2 * H2 * hd2 + D2 * KV2 * hd2 * 2 + H2 * hd2 * D2
ffn_p = 3 * D2 * FF2
total_p = emb_p + L2 * (attn_p + ffn_p + 2 * D2) + D2

print(f"Modified total params: {total_p:,} ~ {total_p/1e6:.1f}M")
print(f"  Embedding: {emb_p:,}")
print(f"  Per layer: {attn_p + ffn_p + 2*D2:,}")
print(f"  {L2} layers: {(attn_p + ffn_p + 2*D2) * L2:,}")

## References

- [SmolLM2-135M model repository](https://huggingface.co/HuggingFaceTB/SmolLM2-135M/tree/main) -- source of all JSON files analyzed in this section
- Touvron et al., [LLaMA: Open and Efficient Foundation Language Models](https://arxiv.org/abs/2302.13971), 2023
- Su et al., [RoFormer: Enhanced Transformer with Rotary Position Embedding](https://arxiv.org/abs/2104.09864), 2021
- Ainslie et al., [GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints](https://arxiv.org/abs/2305.13245), 2023
- Holtzman et al., [The Curious Case of Neural Text Degeneration](https://arxiv.org/abs/1904.09751), 2020 -- original paper on top_p (nucleus sampling)